# Advection Ablation Evaluation

This notebook creates the figure and table for the advection ablation study. 

First set working directory and load the models from the config.

In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.configs.training_config import load_config, build_run_plan
from src.utils.evaluation.evaluation import evaluate_run_test

CONFIG_PATH = Path("configs/training/advection_abblation.yaml")
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config not found: {CONFIG_PATH}")

cfg = load_config(str(CONFIG_PATH))
run_cfgs = build_run_plan(cfg)

print(f"Loaded config: {CONFIG_PATH}")
print(f"Total runs in plan: {len(run_cfgs)}")
pd.DataFrame(run_cfgs).head()

Compute in-domain and out-domain rmse for each model for each run. 

In [ ]:
rows = []
failures = []

for i, run_cfg in enumerate(run_cfgs, start=1):
    run_name = run_cfg["name"]
    model_overrides = run_cfg.get("model_overrides", {})

    # Pull sweep hyperparameters if they exist (late-fusion runs)
    sparsity_weight = model_overrides.get("sparsity_weight", np.nan)
    order_states = model_overrides.get("order_states", np.nan)
    order_parameters = model_overrides.get("order_parameters", np.nan)

    try:
        eval_rows = evaluate_run_test(
            run_cfg=run_cfg,
            domains=("id", "od"),
            use_best=True,
            batch_size_test=500,
        )
        for r in eval_rows:
            rows.append(
                {
                    "run_name": run_name,
                    "model": run_cfg["model"],
                    "seed": run_cfg.get("seed", np.nan),
                    "domain": r.get("domain"),
                    "rmse": r.get("test_rmse", np.nan),
                    "sparsity_weight": sparsity_weight,
                    "order_states": order_states,
                    "order_parameters": order_parameters,
                }
            )
        print(f"[{i}/{len(run_cfgs)}] OK: {run_name}")
    except Exception as e:
        failures.append({"run_name": run_name, "error": str(e)})
        print(f"[{i}/{len(run_cfgs)}] FAILED: {run_name} -> {e}")

raw_df = pd.DataFrame(rows)
fail_df = pd.DataFrame(failures)

print(f"\nSuccessful run-domain evaluations: {len(raw_df)}")
print(f"Failed runs: {len(fail_df)}")

if not fail_df.empty:
    display(fail_df.head(20))

raw_df.head()

Create dataframes with combined errors accross seeds.

In [ ]:

table = (
    raw_df.pivot_table(
        index=[
            "run_name",
            "model",
            "seed",
            "sparsity_weight",
            "order_states",
            "order_parameters",
        ],
        columns="domain",
        values="rmse",
        aggfunc="mean",
    )
    .reset_index()
    .rename(columns={"id": "rmse_id", "od": "rmse_od"})
)

# Keep columns in the requested order
for c in ["rmse_id", "rmse_od"]:
    if c not in table.columns:
        table[c] = np.nan

table = table[[
    "run_name",
    "model",
    "seed",
    "sparsity_weight",
    "order_states",
    "order_parameters",
    "rmse_id",
    "rmse_od",
]]

table = table.sort_values(["model", "sparsity_weight", "order_states", "order_parameters", "seed"], na_position="last").reset_index(drop=True)
display(table)

summary = (
    table.groupby(["model", "sparsity_weight", "order_states", "order_parameters"], dropna=False, as_index=False)
    .agg(
        rmse_id_mean=("rmse_id", "mean"),
        rmse_id_std=("rmse_id", "std"),
        rmse_od_mean=("rmse_od", "mean"),
        rmse_od_std=("rmse_od", "std"),
        n_seeds=("seed", "count"),
    )
    .sort_values(["model", "rmse_od_mean"], na_position="last")
    .reset_index(drop=True)
)

display(summary)

Save results to csv.

In [ ]:
out_dir = Path("outputs/advection_abblation1")
out_dir.mkdir(parents=True, exist_ok=True)

table_path = out_dir / "ablation_rmse_per_run.csv"
summary_path = out_dir / "ablation_rmse_summary.csv"
fail_path = out_dir / "ablation_failed_runs.csv"

table.to_csv(table_path, index=False)
summary.to_csv(summary_path, index=False)
if not fail_df.empty:
    fail_df.to_csv(fail_path, index=False)

print(f"Saved per-run table: {table_path}")
print(f"Saved summary table: {summary_path}")
if not fail_df.empty:
    print(f"Saved failures: {fail_path}")

Create figure for different libraries.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

if table.empty:
    raise RuntimeError("`table` is empty. Run previous cells first.")

plot_df = table.copy()

# Convert possible string columns to numeric for plotting on a log x-axis
plot_df["sparsity_weight"] = pd.to_numeric(plot_df["sparsity_weight"], errors="coerce")
plot_df["rmse_id"] = pd.to_numeric(plot_df["rmse_id"], errors="coerce")
plot_df["rmse_od"] = pd.to_numeric(plot_df["rmse_od"], errors="coerce")

plot_df = plot_df.dropna(subset=["sparsity_weight", "order_states", "order_parameters", "rmse_id", "rmse_od"])
if plot_df.empty:
    raise RuntimeError("No rows with valid sparsity/order/RMSE columns found after numeric conversion.")

# Exclude run-seed pairs with very large errors and print them
run_seed_stats = (
    plot_df.groupby(["run_name", "seed", "order_states", "order_parameters", "sparsity_weight"], as_index=False)
    .agg(
        rmse_id=("rmse_id", "max"),
        rmse_od=("rmse_od", "max"),
    )
)
run_seed_stats["max_rmse"] = run_seed_stats[["rmse_id", "rmse_od"]].max(axis=1)
excluded = run_seed_stats[run_seed_stats["max_rmse"] > 500.0].copy()

if not excluded.empty:
    print("Excluding run-seed entries with RMSE > 500:")
    display(
        excluded[
            [
                "run_name",
                "seed",
                "order_states",
                "order_parameters",
                "sparsity_weight",
                "rmse_id",
                "rmse_od",
                "max_rmse",
            ]
        ].sort_values(["max_rmse", "run_name"], ascending=[False, True]).reset_index(drop=True)
    )

bad_keys = set(tuple(x) for x in excluded[["run_name", "seed"]].drop_duplicates().to_numpy())
if bad_keys:
    plot_df = plot_df[~plot_df.apply(lambda r: (r["run_name"], r["seed"]) in bad_keys, axis=1)].copy()
    print(f"Excluded {len(bad_keys)} run-seed pair(s). Remaining rows: {len(plot_df)}")
else:
    print("No run-seed pairs exceeded RMSE > 500.")

if plot_df.empty:
    raise RuntimeError("All rows were excluded by the RMSE > 500 filter.")

# Mean and std over seeds for each (complexity, sparsity)
agg = (
    plot_df.groupby(["order_states", "order_parameters", "sparsity_weight"], as_index=False)
    .agg(
        rmse_id_mean=("rmse_id", "mean"),
        rmse_id_std=("rmse_id", "std"),
        rmse_od_mean=("rmse_od", "mean"),
        rmse_od_std=("rmse_od", "std"),
    )
    .sort_values(["order_states", "order_parameters", "sparsity_weight"])
)

# If only one seed is available, std is NaN -> show as 0
for c in ["rmse_id_std", "rmse_od_std"]:
    agg[c] = agg[c].fillna(0.0)

combos = (
    agg[["order_states", "order_parameters"]]
    .drop_duplicates()
    .sort_values(["order_states", "order_parameters"])
    .values
    .tolist()
)

library_titles = {
    (1, 1): "Library 1 (6 terms)",
    (1, 2): "Library 2 (9 terms)",
    (2, 1): "Library 3 (12 terms)",
    (2, 2): "Library 4 (18 terms)",
}

FONTSIZE = 9
n = len(combos)
ncols = 2
nrows = (n + ncols - 1) // ncols
fig_height = 3 * nrows
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(10, fig_height), squeeze=False)
axes_flat = axes.flatten()

# Requested x-ticks in descending order from left to right on a log axis
tick_values = [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-7]
tick_labels = ["1e-1", "1e-2", "1e-3", "1e-4", "1e-5", "1e-6", "1e-7"]

# Same palette as parameter plots (mean point colors + edges)
id_face, id_edge = "#5face4", "#27516e"
od_face, od_edge = "#45792D", "#1a380d"
err_gray = "#8a8a8a"

for i, (os_, op_) in enumerate(combos):
    ax = axes_flat[i]
    sub = agg[(agg["order_states"] == os_) & (agg["order_parameters"] == op_)].copy()
    sub = sub.sort_values("sparsity_weight", ascending=False)

    sub_id = sub.dropna(subset=["sparsity_weight", "rmse_id_mean"])
    sub_od = sub.dropna(subset=["sparsity_weight", "rmse_od_mean"])

    if not sub_id.empty:
        ax.errorbar(
            sub_id["sparsity_weight"],
            sub_id["rmse_id_mean"],
            yerr=sub_id["rmse_id_std"],
            fmt="s-",
            markersize=7,
            markerfacecolor=id_face,
            markeredgecolor=id_edge,
            markeredgewidth=1.0,
            color=id_edge,
            ecolor=err_gray,
            elinewidth=1.1,
            capsize=3,
            linewidth=1.2,
            zorder=3,
        )

    if not sub_od.empty:
        ax.errorbar(
            sub_od["sparsity_weight"],
            sub_od["rmse_od_mean"],
            yerr=sub_od["rmse_od_std"],
            fmt="s-",
            markersize=7,
            markerfacecolor=od_face,
            markeredgecolor=od_edge,
            markeredgewidth=1.0,
            color=od_edge,
            ecolor=err_gray,
            elinewidth=1.1,
            capsize=3,
            linewidth=1.2,
            zorder=3,
        )

    ax.set_xscale("log")
    ax.set_xlabel(r"$\lambda_{\mathrm{sparse}}$", fontsize=FONTSIZE)
    ax.set_ylabel("RMSE", fontsize=FONTSIZE)
    ax.set_title(library_titles.get((int(os_), int(op_)), f"order_states={int(os_)}, order_parameters={int(op_)}"), fontsize=FONTSIZE)
    ax.grid(True, alpha=0.3)

    ax.set_xticks(tick_values)
    ax.set_xticklabels(tick_labels)
    ax.set_xlim(1e-0, 1e-8)
    ax.tick_params(axis="both", labelsize=FONTSIZE)

    # Keep y-axis independent per subplot
    yvals = np.concatenate([
        sub[["rmse_id_mean", "rmse_od_mean"]].to_numpy().reshape(-1),
        (sub[["rmse_id_mean"]].to_numpy().reshape(-1) + sub[["rmse_id_std"]].to_numpy().reshape(-1)),
        (sub[["rmse_od_mean"]].to_numpy().reshape(-1) + sub[["rmse_od_std"]].to_numpy().reshape(-1)),
    ])
    yvals = yvals[~pd.isna(yvals)]
    if yvals.size > 0:
        y_min = float(yvals.min())
        y_max = float(yvals.max())
        if y_max > y_min:
            pad = 0.08 * (y_max - y_min)
            ax.set_ylim(y_min - pad, y_max + pad)

# Hide unused subplot axes
for j in range(n, len(axes_flat)):
    axes_flat[j].axis("off")

# One shared legend at the bottom, like parameter plot
legend_handles = [
    mpl.lines.Line2D([0], [0], marker="s", color=id_edge, markerfacecolor=id_face, markeredgecolor=id_edge, markersize=7, linestyle="-", linewidth=1.2),
    mpl.lines.Line2D([0], [0], marker="s", color=od_edge, markerfacecolor=od_face, markeredgecolor=od_edge, markersize=7, linestyle="-", linewidth=1.2),
]
legend_labels = ["In-domain", "Out-domain"]

fig.subplots_adjust(bottom=0.24)  # reserve space for legend area
legend_ax = fig.add_axes([0.40, 0.05, 0.22, 0.10])  # y moved up: 0.01 -> 0.05
legend_ax.axis("off")
legend_ax.legend(
    legend_handles,
    legend_labels,
    loc="center",
    ncol=2,
    framealpha=0.2,
    fontsize=FONTSIZE,
)

# No figure title per request
fig.tight_layout(rect=[0, 0.10, 1, 0.98])
plt.show()

fig_path = Path("outputs/rmse_vs_sparsity_by_complexity.pdf")
fig_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_path, format="pdf", bbox_inches="tight")
print(f"Saved figure: {fig_path}")